In [1]:
# Cell 1: Check whether the server and model survived
!ollama list

/bin/bash: line 1: ollama: command not found


In [2]:
# Cell 2: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
# Cell 3: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,583 kB of archives.
After this operation, 5,343 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1 [334 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble/main amd64 usb.ids all 2024.03.18-1 [223 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-update

In [4]:
# Cell 4: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# Cell 5: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

In [6]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

In [7]:
# Cell 7: Download the text model
import ollama

print(ollama.pull("gemma4:e2b").status)

success


In [8]:
# Cell 8: Download the embedding model
import ollama

print(ollama.pull("nomic-embed-text").status)

success


In [9]:
# Cell 9: Turn one sentence into a vector

response = ollama.embed(
    model = "nomic-embed-text",
    input = "Cable glands seal the point where a cable enters an enclosure."
)

vector = response.embeddings[0]
print(len(vector))
print(vector[:5])

768
[0.030740838, 0.05348693, -0.14595363, -0.055521727, 0.02450215]


In [10]:
# Cell 10: Compare two sentences with cosine similarity

import math 
sentences = [
    "Cable glands seal the point where a cable enters an enclosure.",
    "How do I close the gap where a wire goes into a box?",
    "A busbar carries current between circuit breakers in a panel.",
]

# One call can embed several sentences; the vectors come back in the same order
vectors = ollama.embed(model = "nomic-embed-text", input = sentences).embeddings


def cosine (a, b):
    """Return the cosine similarity of two vectors (1 = same direction)."""
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot / (length_a * length_b)

print("0 vs 1:", round(cosine(vectors[0], vectors[1]), 3))
print("0 vs 2:", round(cosine(vectors[0], vectors[2]), 3))
print("1 vs 2:", round(cosine(vectors[1], vectors[2]), 3))

0 vs 1: 0.574
0 vs 2: 0.495
1 vs 2: 0.532


In [11]:
# Cell 11: Cosine similarity with prints, on three-number vectors
import math


def cosine_verbose(a, b):
    """Return the cosine similarity of two vectors, printing each step."""
    print("a =", a)
    print("b =", b)

    # Pair the numbers: first with first, second with second, and so on
    pairs = list(zip(a, b))
    print("pairs (x, y) =", pairs)

    # Multiply each pair, then add the products
    products = [x * y for x, y in pairs]
    dot = sum(products)
    print("products x*y =", products)
    print("dot =", dot)

    # The size of each vector: square every number, add, take the square root
    squares_a = [x * x for x in a]
    squares_b = [y * y for y in b]
    length_a = math.sqrt(sum(squares_a))
    length_b = math.sqrt(sum(squares_b))
    print("squares of a =", squares_a, "-> length_a =", round(length_a, 3))
    print("squares of b =", squares_b, "-> length_b =", round(length_b, 3))

    # Divide by both sizes so only the direction counts
    score = dot / (length_a * length_b)
    print("score = dot / (length_a * length_b) =", round(score, 3))
    return score


cosine_verbose([1, 2, 3], [3, 2, 1])

a = [1, 2, 3]
b = [3, 2, 1]
pairs (x, y) = [(1, 3), (2, 2), (3, 1)]
products x*y = [3, 4, 3]
dot = 10
squares of a = [1, 4, 9] -> length_a = 3.742
squares of b = [9, 4, 1] -> length_b = 3.742
score = dot / (length_a * length_b) = 0.714


0.7142857142857143

In [21]:
# Cell 12: Rank four notes for one question
# Four invented notes. The ids are the sources we will show in answers later.

notes = {
    "D1": "A moulded case circuit breaker switches off a circuit when the current is too high.",
    "D2": "A busbar carries current between circuit breakers in a panel.",
    "D3": "Cable glands seal the point where a cable enters an enclosure.",
    "D4": "A current transformer measures the current in a cable so that a meter can read it.",
}

# Embed all four notes in one call; the vectors keep the same order as the notes
note_vectors = ollama.embed(
    model = "nomic-embed-text",
    input = ["search_document: " + text for text in notes.values()]
).embeddings

# Embed the question in the same way
question = " How do I close the gap where a wire goes into a box?"
question_vector = ollama.embed(
    model = "nomic-embed-text",
    input = "search_query: " + question
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores = []
for note_id, note_vector in zip(notes.keys(), note_vectors):
    scores.append((cosine(question_vector, note_vector), note_id))

# Highest score first (sort orders pairs by their first item, the score)
scores.sort(reverse=True)

for score, note_id in scores:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.583
D1: 0.582
D3: 0.581
D4: 0.512


In [ ]:
# Cell 12b: Rank four notes for one question
# Four invented notes. The ids are the sources we will show in answers later.

notes = {
    "D1": "A moulded case circuit breaker switches off a circuit when the current is too high.",
    "D2": "A busbar carries current between circuit breakers in a panel.",
    "D3": "Cable glands seal the point where a cable enters an enclosure.",
    "D4": "A current transformer measures the current in a cable so that a meter can read it.",
}

# Embed all four notes in one call; the vectors keep the same order as the notes
note_vectors = ollama.embed(
    model = "nomic-embed-text",
    input = ["search_document: " + text for text in notes.values()]
).embeddings

# Embed the question in the same way
question = "Which part carries current between breakers in a panel?" # The question is differnt in this version.
question_vector = ollama.embed(
    model = "nomic-embed-text",
    input = "search_query: " + question
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores = []
for note_id, note_vector in zip(notes.keys(), note_vectors):
    scores.append((cosine(question_vector, note_vector), note_id))

# Highest score first (sort orders pairs by their first item, the score)
scores.sort(reverse=True)

for score, note_id in scores:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.833
D1: 0.67
D4: 0.635
D3: 0.524


This is a clear win. D2 scores 0.833, and the next note is 0.16 behind. So the model works when the question shares words with a note. It struggles when the question uses different words, as in the “wire in a box” question.

Question	Winner	Gap to second place
Easy: “Which part carries current between breakers in a panel?”	D2 (correct)	0.163
Hard: “How do I close the gap where a wire goes into a box?”	D2 (wrong)	0.002

Now we test a stronger embedding model on the same two questions in **cell 15**

The call returns one reply object. It holds the vectors and some extra details. .embeddings picks out the vectors and drops the rest. These two versions do the same job:

```py

# Version A: keep the whole reply, then pick the vectors out of it
response = ollama.embed(model="nomic-embed-text", input=list(notes.values()))
note_vectors = response.embeddings

# Version B: pick them out straight away (what Cell 12 does)
note_vectors = ollama.embed(model="nomic-embed-text", input=list(notes.values())).embeddings
```

```
response                 the whole reply
├── model                the model name
├── ...                  other details
└── embeddings           the list of vectors  ← this is note_vectors
    ├── [0]              vector for D1 (768 numbers)
    ├── [1]              vector for D2
    ├── [2]              vector for D3
    └── [3]              vector for D4

```
An everyday picture: the reply is a parcel. .embeddings takes out the one item you want. You throw the box away.

In [15]:
 # Cell 13: Compare the whole reply with its embeddings part
response = ollama.embed(model="nomic-embed-text", input=list(notes.values()))

print(type(response))
print(type(response.embeddings))
print(len(response.embeddings))
print(len(response.embeddings[0]))

print("response.embeddings",(response.embeddings))
print("response.embeddings[0]",(response.embeddings[0]))

<class 'ollama._types.EmbedResponse'>
<class 'list'>
4
768
response.embeddings [[0.067672305, -0.0007982441, -0.18156846, -0.0222183, 0.04234001, 0.027774401, 0.01700524, 0.06784547, 0.0048330436, -0.025513489, -0.07925224, 0.028610393, 0.016621089, 0.025608465, 0.011143727, -0.025729472, -0.030427322, -0.0537656, 0.03425565, -0.0018445612, 3.2492997e-05, -0.028320033, -0.021294579, 0.011256345, 0.017249849, 0.034162432, 0.055653192, 0.03388824, -0.007717917, 0.047399852, -0.024561584, -0.0063853106, 0.029224327, -0.07899269, -0.09929106, -0.06226927, 0.060499713, 0.032142222, 0.009737055, 0.009322659, 0.041123424, 0.02876905, 0.013490235, -0.070939004, 0.06278041, 0.014701837, 0.042823955, 0.01414415, -0.03452474, -0.046768352, -0.003746001, -0.034269836, 0.0020412705, -0.07083476, 0.065563805, 0.0006958924, 0.03270012, 0.035341445, 0.03505823, -0.016008586, 0.025398526, 0.10131077, 0.012660601, 0.043373402, -0.00017414821, 0.026637452, -0.035111, 0.021316405, 0.015951168, -0.02330195

response.embeddings[0] is the note D1, the whole sentence: “A moulded case circuit breaker switches off a circuit when the current is too high.”
The 768 numbers stand for the meaning of that entire sentence, not for one word in it.
* D1 (13 words)  ->  response.embeddings[0]  ->  768 numbers
* D2 (11 words)  ->  response.embeddings[1]  ->  768 numbers

What happens inside: the embedding model reads all the words together, including their order and context. It then gives back one list of 768 numbers for the whole text. There are no per-word numbers in the output.

An everyday picture: think of the 768 numbers as an address on a map of meaning. A short address would not be enough to tell thousands of meanings apart, so the address has 768 parts. Every text gets one address. Texts with similar meaning get addresses close together.

Two facts follow:

* Every note gets exactly 768 numbers, long or short.
* The individual numbers mean nothing alone, such as 0.0677 or -0.0008. Only the whole address, compared with other addresses, carries meaning.

In [16]:
# Cell 14: Check that any length of text gives one vector of the same size
short = ollama.embed(model="nomic-embed-text", input="busbar").embeddings
long = ollama.embed(model="nomic-embed-text", input=notes["D1"] + " " + notes["D2"]).embeddings

print("one word:     ", len(short), "vector(s) of", len(short[0]), "numbers")
print("two sentences:", len(long), "vector(s) of", len(long[0]), "numbers")

one word:      1 vector(s) of 768 numbers
two sentences: 1 vector(s) of 768 numbers


The size is built into the model when it is trained.
```
Embedding model	Numbers per vector
all-MiniLM-L6-v2	384
nomic-embed-text (yours)	768
mxbai-embed-large	1024
OpenAI text-embedding-3-small	1536
OpenAI text-embedding-3-large	3072
```

In [23]:
# Cell 15: Download a larger embedding model (1024 numbers per vector)
import ollama

print(ollama.pull("mxbai-embed-large").status)

success


In [25]:
# Cell 16a: Rank the same four notes with the larger embedding model

# All four note vectors (no [0]), embedded by the larger model
note_vectors_large = ollama.embed(
    model="mxbai-embed-large",  # larger model, 1024 numbers per vector
    input=list(notes.values()),
).embeddings

# One question, one vector ([0] takes it out of the list of one)
question = "How do I close the gap where a wire goes into a box?"
question_vector_large = ollama.embed(
    model="mxbai-embed-large",
    input=question,
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores_large = []
for note_id, note_vector in zip(notes.keys(), note_vectors_large):
    scores_large.append((cosine(question_vector_large, note_vector), note_id))

scores_large.sort(reverse=True)

for score, note_id in scores_large:
    print(f"{note_id}: {round(score, 3)}")

D3: 0.699
D2: 0.577
D1: 0.544
D4: 0.534


In [26]:
# Cell 16b: Rank the same four notes with the larger embedding model

# All four note vectors (no [0]), embedded by the larger model
note_vectors_large = ollama.embed(
    model="mxbai-embed-large",  # larger model, 1024 numbers per vector
    input=list(notes.values()),
).embeddings

# One question, one vector ([0] takes it out of the list of one)
question = "Which part carries current between breakers in a panel?"
question_vector_large = ollama.embed(
    model="mxbai-embed-large",
    input=question,
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores_large = []
for note_id, note_vector in zip(notes.keys(), note_vectors_large):
    scores_large.append((cosine(question_vector_large, note_vector), note_id))

scores_large.sort(reverse=True)

for score, note_id in scores_large:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.858
D1: 0.703
D4: 0.575
D3: 0.572


Retrieval found the right note. Now the model writes the answer from that note only. The source comes from our code, not from the model. We know which note we handed over, so the source cannot be made up.

In [27]:
# Cell 17: Answer from the best note and show its source

# The top pair is (score, note id)

best_score, best_id = scores_large[0]
best_note = notes[best_id]

response = ollama.chat(
    model = "gemma4:e2b",
    messages = [
        {
            "role": "system",
            "content": ("Answer the question using the note provided. "
                        "If the note does not contain the answer, say 'I don't know.'")
        },
        {
            "role": "user",
            "content": f"Note:\n{best_note}\n\nQuestion:\n{question}"
        },
    ],
    think = False,
)
print(response.message.content)
print(f"Source: {best_id} (score {round(best_score, 3)})")

A busbar carries current between circuit breakers in a panel.
Source: D2 (score 0.858)


In [29]:
# Cell 18: Ask several questions in a loop

def answer_from_notes(question):
    """Find the best note for a question and answer from it.
    Returns the answer text, the id of the note used (None if refused), and the best score."""
    # Embed the question with the same model used for the notes
    question_vector = ollama.embed(
        model="mxbai-embed-large",
        input=question,
    ).embeddings[0]

    # Score every note; each pair is (score, note id)
    scores = []
    for note_id, note_vector in zip(notes.keys(), note_vectors_large):
        scores.append((cosine(question_vector, note_vector), note_id))
    scores.sort(reverse=True)
    best_score, best_id = scores[0]

    # Answer from the best note only
    response = ollama.chat(
        model="gemma4:e2b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question using only the note provided. "
                    "If the note does not contain the answer, say: I cannot answer from the notes."
                ),
            },
            {
                "role": "user",
                "content": f"Note:\n{notes[best_id]}\n\nQuestion: {question}",
            },
        ],
        think=False,
    )

    answer = response.message.content
    # A refusal means the note did not support an answer, so no source is reported
    refused = "cannot answer" in answer.lower()
    source = None if refused else best_id
    return answer, source, best_score


questions = [
    "Which part carries current between breakers in a panel?",
    "How do I close the gap where a wire goes into a box?",
    "What is the price of a busbar?",
    "What lets a meter read the current in a cable?",
]

for q in questions:
    answer, source, score = answer_from_notes(q)
    print("Q:", q)
    print("A:", answer)
    if source is None:
        print(f"Source: none (best note scored {round(score, 3)}, but it did not hold the answer)")
    else:
        print(f"Source: {source} (score {round(score, 3)})")
    print()

Q: Which part carries current between breakers in a panel?
A: A busbar carries current between circuit breakers in a panel.
Source: D2 (score 0.858)

Q: How do I close the gap where a wire goes into a box?
A: Cable glands seal the point where a cable enters an enclosure.
Source: D3 (score 0.699)

Q: What is the price of a busbar?
A: I cannot answer from the notes.
Source: none (best note scored 0.609, but it did not hold the answer)

Q: What lets a meter read the current in a cable?
A: A current transformer measures the current in a cable so that a meter can read it.
Source: D4 (score 0.913)

